In [ ]:
# %%
from pathlib import Path
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
print("Project root:", project_root)
from eintelligence.data_prep.aoi import square_aoi
from orchestrator.workflow_manager_multisensor_v1 import (
    DeforestationWorkflowMS, TilingConfigMS, TilingConfigS1, TrainingConfig,
    FloodWorkflowS1
)


CASE = 1

# 0 - DEFORESTATION - S2 
# 1 - DEFORESTSTION - MULTISENSOR
# 2 - FLOOD - S1



In [ ]:
aoi = square_aoi(48.1351, 11.5820)
# aoi = square_aoi(-7.754, -55.513)  # Novo Progresso (Pará, BR-163 / Jamanxim front)

# Pick ONE:
# sensor_mode = "s2"
# sensor_mode = "s1"
sensor_mode = "s1s2"

if CASE == 1:
    case_name = "deforestation"
    
    tiling_cfg = TilingConfigMS(
        bands_s2=("B02","B03","B04","B08"),
        bands_s1=("vv","vh"),
        tile_size=256, stride=256, max_cloud=50,
        sensor_mode=sensor_mode
    )

    train_cfg  = TrainingConfig(batch_size=4, num_epochs=10, lr=1e-3, amp=True)

    wf = DeforestationWorkflowMS(project_root, tiling_cfg, train_cfg, skip_to_pairing=False)

elif CASE == 2:
    case_name = "flood"
    tiling_cfg = TilingConfigS1(tile_size=256, stride=256)

    train_cfg  = TrainingConfig(batch_size=4, num_epochs=10, lr=1e-3, amp=True)
    
    wf = FloodWorkflowS1(project_root, tiling_cfg, train_cfg, skip_to_pairing=False)

print("The case is - ", case_name)


In [ ]:


region_name = f"munich_{case_name}_{sensor_mode}"
pairs_manifest = wf.build_data(
    aoi_geojson=aoi,
    start="2023-06-01",
    end="2023-08-01",
    region_name=region_name
)

ckpt_path = Path(project_root) / "models" / f"{case_name}_{sensor_mode}_adapter.pt"
out_dir   = Path(project_root) / "data" / region_name / f"pred_{case_name}_{sensor_mode}"

wf.run(pairs_manifest, ckpt_path, out_dir, retrain=True, prob_thresh=0.5)